<a href="https://colab.research.google.com/github/soumaya-lamrhari/customer-conversion-prediction/blob/main/notebooks/1_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Conversion Prediction

## Objective

In this project, I use customer and marketing campaign data to predict whether a customer will subscribe to a term deposit.

The main goal is to identify customers with a higher likelihood of conversion, so that marketing efforts can be focused on those most likely to convert.

## Business Question

Which customers are most likely to subscribe to a term deposit, and how can the bank use these predictions to prioritize customers for marketing outreach?


In [1]:
!pip install -q ucimlrepo

In [2]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

In [5]:
# Load the Bank Marketing dataset from the UCI repository
bank_marketing = fetch_ucirepo(id=222)
# Separate features and target for modeling
X = bank_marketing.data.features
y = bank_marketing.data.targets

**Data Exploration**

Before building the model, I first explore the dataset to understand its structure, check data quality, and examine the conversion outcome.

In [6]:
# Check the size of the feature set and target before starting the analysis
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (45211, 16)
Target shape: (45211, 1)


In [7]:
# Review the available variables before deciding how to prepare them for modeling
print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.columns.tolist())

Feature columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']

Target column:
['y']


In [8]:
# Preview a few records to understand how customer and campaign information is represented
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [9]:
# Check data types to understand which features are numerical and categorical
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


In [10]:
# Quantify missing values before deciding how they should be handled
missing_values = pd.DataFrame({
    "missing_count": X.isnull().sum(),
    "missing_percentage": (X.isnull().mean() * 100).round(2)
})

missing_values[missing_values["missing_count"] > 0].sort_values(
    "missing_percentage", ascending=False
)

,missing_count,missing_percentage
poutcome,36959,81.75
contact,13020,28.80
education,1857,4.11
job,288,0.64


In [11]:
# Inspect categories in features with missing values before making any cleaning decision
for column in ["poutcome", "contact", "education", "job"]:
    print(f"\n{column}:")
    print(X[column].value_counts(dropna=False))


poutcome:
poutcome
NaN        36959
failure     4901
other       1840
success     1511
Name: count, dtype: int64

contact:
contact
cellular     29285
NaN          13020
telephone     2906
Name: count, dtype: int64

education:
education
secondary    23202
tertiary     13301
primary       6851
NaN           1857
Name: count, dtype: int64

job:
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
NaN               288
Name: count, dtype: int64
